# 📓 Notebook 5｜特徵子集選擇：順序前向選擇 SFS（站 5）

> 對應講義 Part 5（5.7）。從 l 個特徵挑出最好的 k 個子集。窮舉 $2^l$ 太貴，
> 用貪婪法逼近：**順序前向選擇 SFS** 每步加「讓準則函數增最多」的特徵。

In [ ]:
import numpy as np
from itertools import combinations
rng = np.random.default_rng(0)

# 合成 8 個特徵，其中前 3 個有區辨力，後 5 個是雜訊
N = 100
X1 = rng.normal(0, 1, (N, 8)); X2 = rng.normal(0, 1, (N, 8))
X1[:, :3] += 1.0          # 前 3 個特徵有區辨力（均值差 1）
X = np.vstack([X1, X2]); y = np.array([0] * N + [1] * N)

def criterion(cols):
    '''用 FDR 總和當準則（越大越好）。'''
    s = 0.0
    for c in cols:
        s += (X1[:, c].mean() - X2[:, c].mean()) ** 2 / (X1[:, c].var() + X2[:, c].var())
    return s

print('單一特徵的 FDR（前 3 個應明顯較大）:')
for c in range(8):
    print(f'  特徵 {c}: FDR = {criterion([c]):.3f}')

### 順序前向選擇 SFS

從空集合開始，每步加入「讓準則增最多」的那個特徵，直到選滿 k 個。

In [ ]:
def sfs(k):
    selected = []
    remaining = list(range(8))
    for _ in range(k):
        best, best_gain = None, -1
        for c in remaining:
            gain = criterion(selected + [c])
            if gain > best_gain:
                best, best_gain = c, gain
        selected.append(best); remaining.remove(best)
        print(f'  加入特徵 {best}，準則 = {best_gain:.3f}')
    return selected

print('SFS 選出前 4 個特徵:')
sel = sfs(4)
print('→ 選出的特徵:', sel, '（應包含前 3 個有區辨力的）')

### 浮動搜索（加一步、退一步）

SFS 只進不退，可能卡在局部。浮動法允許「加完後，若移除某個已選特徵能提升準則，就退掉它」。

In [ ]:
def sffs(k):
    selected = []
    remaining = list(range(8))
    while len(selected) < k:
        # 加：選增益最大的
        best, best_gain = None, -1
        for c in remaining:
            g = criterion(selected + [c])
            if g > best_gain: best, best_gain = c, g
        selected.append(best); remaining.remove(best)
        # 退：若移除某個已選特徵能提升準則，就退掉
        if len(selected) > 2:
            for c in selected:
                trial = [s for s in selected if s != c]
                if criterion(trial) > criterion(selected):
                    selected = trial; remaining.append(c); break
    return selected

print('浮動搜索選出前 4 個特徵:', sffs(4))

# ✏️ 練習：把 k 改成 6，看是否會把雜訊特徵也選進來
